In [ ]:
# =============================================================================
# ЯЧЕЙКА 0: Проверка доступности LLM API
# Провайдер читается из .env → PROVIDER=yandex|deepseek|qwen3
# =============================================================================
import os
import logging
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

# FIX 3.1: явный путь — CWD внутри ноутбука = /workspace/notebooks,
# без аргумента load_dotenv() не найдёт /workspace/.env
load_dotenv("/workspace/.env")


def get_llm() -> ChatOpenAI:
    """
    Фабрика LLM: читает PROVIDER из env, возвращает ChatOpenAI.
    Все три провайдера используют OpenAI-совместимый endpoint.

    Raises:
        ValueError: если PROVIDER неизвестен или отсутствуют обязательные переменные.
    """
    provider = os.getenv("PROVIDER", "yandex").lower()

    if provider == "yandex":
        api_key   = os.getenv("YANDEX_API_KEY")
        folder_id = os.getenv("YANDEX_FOLDER_ID")
        if not api_key or not folder_id:
            raise ValueError("Не заданы YANDEX_API_KEY или YANDEX_FOLDER_ID в .env")
        return ChatOpenAI(
            base_url="https://llm.api.cloud.yandex.net/v1",
            api_key=api_key,
            model=f"gpt://{folder_id}/yandexgpt/latest",
            # x-folder-id обязателен для YandexGPT — нестандартный OpenAI заголовок
            default_headers={"x-folder-id": folder_id},
            temperature=0.7,
            max_tokens=200,
        )

    elif provider == "deepseek":
        api_key = os.getenv("DEEPSEEK_API_KEY")
        if not api_key:
            raise ValueError("Не задан DEEPSEEK_API_KEY в .env")
        return ChatOpenAI(
            base_url="https://api.deepseek.com/v1",
            api_key=api_key,
            model="deepseek-chat",
            temperature=0.7,
            max_tokens=200,
        )

    elif provider == "qwen3":
        # Ollama на хосте → host.docker.internal работает благодаря extra_hosts в compose
        return ChatOpenAI(
            base_url="http://host.docker.internal:11434/v1",
            api_key="EMPTY",  # Ollama/vLLM требуют непустую строку
            model="qwen3:8b",
            temperature=0.7,
            max_tokens=200,
        )

    else:
        raise ValueError(
            f"Неизвестный PROVIDER='{provider}'. Допустимо: yandex | deepseek | qwen3"
        )


# Создаём один раз — llm доступен во всех следующих ячейках
llm = get_llm()

response = llm.invoke([HumanMessage(content="Кто ты? Ответь одним предложением.")])

print(f"✅ API доступен")
print(f"   Провайдер : {os.getenv('PROVIDER', 'yandex').upper()}")
print(f"   Ответ     : {response.content}")


In [ ]:
# ── ЯЧЕЙКА 1 ──────────────────────────────────────────────────
# ⏱ ~2 мин | 📌 Проверяем GPU, пакеты, диск
# ⚠️ Если GPU не видно — проверить что Pod запущен с GPU шаблоном
# ⚠️ Если BF16 = False — использовать float16 в следующих ячейках

import subprocess, torch

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

print("=" * 60)
print(f"PyTorch:       {torch.__version__}")
print(f"CUDA OK:       {torch.cuda.is_available()}")
print(f"GPU:           {torch.cuda.get_device_name(0)}")
total = torch.cuda.get_device_properties(0).total_memory / 1024**3
used  = torch.cuda.memory_reserved(0) / 1024**3
print(f"VRAM:               {total:.1f} GB всего, {total-used:.1f} GB свободно")
print(f"BF16 support:  {torch.cuda.is_bf16_supported()}")

# Проверяем диск
disk_info = subprocess.run(['df','-h','/workspace'], capture_output=True, text=True)
print(f"\nДиск /workspace:    {disk_info.stdout.split()[-4]} свободно")


In [ ]:
# ── ЯЧЕЙКА 2 ──────────────────────────────────────────────────────────────
# ⏱ ~1 мин | 📌 Проверяем что Ollama работает
# ⚠️ Ollama запущен в /pre_start.sh как background process

import requests, time

def wait_for_ollama(max_attempts=10):
    for i in range(max_attempts):
        try:
            r = requests.get("http://localhost:11434/api/tags", timeout=5)
            if r.status_code == 200:
                models = r.json().get("models", [])
                print(f"Ollama OK ✅ — доступных моделей: {len(models)}")
                for m in models:
                    size_gb = m.get('size', 0) / 1024**3
                    print(f"  - {m['name']} ({size_gb:.1f} ГБ)")
                return True
        except Exception as e:
            print(f"Попытка {i+1}/{max_attempts}: Ollama ещё стартует...")
            time.sleep(3)
    return False

if not wait_for_ollama():
    print("⚠️ Ollama не ответила. Запустим вручную:")
    print("! OLLAMA_MODELS=/workspace/ollama_models ollama serve &")


In [ ]:
# ── ЯЧЕЙКА 3 ──────────────────────────────────────────────────────────────
# ⏱ ~2 мин | 📌 Первый запрос к локальной LLM через OpenAI API
# ⚠️ Показать: тот же API что у OpenAI, только base_url другой

import requests, json

response = requests.post(
    "http://localhost:11434/v1/chat/completions",
    headers={"Content-Type": "application/json"},
    json={
        "model": "qwen3:8b",
        "messages": [
            {"role": "system", "content": "Отвечай кратко и по делу."},
            {"role": "user", "content": "Ты локальная или облачная модель? Откуда ты работаешь?"}
        ]
    }
)

data = response.json()
print(f"Ответ: {data['choices'][0]['message']['content']}")
print(f"Токены: input={data['usage']['prompt_tokens']}, output={data['usage']['completion_tokens']}")


In [ ]:
# ── ЯЧЕЙКА 4 ──────────────────────────────────────────────────────────────
# ⏱ ~1 мин | 📌 Структура /workspace — где что лежит

import os, subprocess

def show_workspace():
    for item in sorted(os.listdir("/workspace")):
        path = f"/workspace/{item}"
        if os.path.isdir(path):
            size = subprocess.run(['du', '-sh', path],
                capture_output=True, text=True).stdout.split()[0]
            print(f"  📁 {item}/  ({size})")
        else:
            size = os.path.getsize(path) / 1024**2
            print(f"  📄 {item}  ({size:.1f} МБ)")

print("/workspace:")
show_workspace()
